# Data Center Drought Risk - Interactive Map

Visualises every data center in `DataCenter_USGS_Drought_Forecast.csv` on a Folium map.  
Click any marker to see a **monthly drought intensity chart** (styled after Figure 1 in the Terra_Project README)  
showing the median streamflow percentile of the nearest USGS gage, with 5-95% prediction intervals.

**Risk colour scheme** (matches PDSI convention):
| Risk Status | Colour |
|---|---|
| Extreme Risk | Brown |
| High Risk | Orange |
| Warning | Amber |
| Safe / Normal | Green |

In [ ]:
import warnings, base64, io

warnings.filterwarnings("ignore")

import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import folium
from folium.plugins import MarkerCluster

print("Libraries loaded OK")

In [ ]:
CSV = "DataCenter_USGS_Drought_Forecast.csv"
df = pd.read_csv(CSV)

# Dates are DD-MM-YY  (e.g. 19-04-26 = 19 Apr 2026)
df["date"] = pd.to_datetime(df["forecast_date"], format="%d-%m-%y")
df["year_month"] = df["date"].dt.to_period("M")

for col in ["median_pct", "pred_interv_05_pct", "pred_interv_95_pct"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(f"Loaded {len(df):,} rows  |  {df['Name'].nunique():,} unique data centers")
print(f"Date range : {df['date'].min().date()}  to  {df['date'].max().date()}")
print(f"Risk levels: {sorted(df['Risk_Status'].dropna().unique())}")

In [ ]:
RISK_ORDER = {
    "Extreme Risk": 0,
    "High Risk": 1,
    "Warning": 2,
    "Watch": 3,
    "Safe": 4,
    "Normal": 5,
    "Wet": 6,
}


def dominant_risk(series):
    """Return the worst (most severe) risk status observed in the period."""
    ranked = series.map(lambda r: RISK_ORDER.get(r, 99))
    return series.iloc[ranked.argmin()]


meta_cols = [
    "Name",
    "Operator",
    "State",
    "City",
    "Power",
    "Latitude",
    "Longitude",
    "nearest_station_id",
    "distance_km",
]

monthly = df.groupby(meta_cols + ["year_month"], as_index=False).agg(
    median_pct=("median_pct", "median"),
    ci_05=("pred_interv_05_pct", "median"),
    ci_95=("pred_interv_95_pct", "median"),
    Risk_Status=("Risk_Status", dominant_risk),
    n_weeks=("forecast_date", "count"),
)

monthly["month_label"] = monthly["year_month"].dt.strftime("%b %Y")
monthly["month_dt"] = monthly["year_month"].dt.to_timestamp()
monthly = monthly.sort_values(["Name", "month_dt"])

print(f"Monthly records: {len(monthly):,}")
monthly[["Name", "month_label", "median_pct", "Risk_Status"]].head(8)

In [ ]:
# Bar / fill colours  (dark background, PDSI-aligned palette)
RISK_COLORS = {
    "Extreme Risk": "#6B2A00",  # deep brown
    "High Risk": "#E05C00",  # burnt orange
    "Warning": "#E6A817",  # amber
    "Watch": "#CCCC00",  # olive-yellow
    "Safe": "#6AAA64",  # muted green
    "Normal": "#6AAA64",
    "Wet": "#1D7A3C",  # deep green
}
DEFAULT_COLOR = "#888888"

# Circle marker colours on the map (same palette, slightly brighter)
MARKER_COLORS = {
    "Extreme Risk": "#8B3A00",
    "High Risk": "#FF6B00",
    "Warning": "#FFD700",
    "Watch": "#CCCC00",
    "Safe": "#4CAF50",
    "Normal": "#4CAF50",
    "Wet": "#1B7E35",
}


def worst_risk(group):
    """Return the overall worst risk label for a data center."""
    ranked = group["Risk_Status"].map(lambda r: RISK_ORDER.get(r, 99))
    return group.loc[ranked.idxmin(), "Risk_Status"]


print("Colour scheme defined OK")

In [ ]:
def make_chart_b64(name: str, group: pd.DataFrame) -> str:
    """
    Build a Figure-1-style monthly drought bar chart for the available data months.
    Returns a base64-encoded PNG for embedding in an HTML popup.
    """
    group = group.sort_values("month_dt").reset_index(drop=True)
    month_labels = group["month_label"].tolist()
    pct = group["median_pct"].tolist()
    ci05 = group["ci_05"].fillna(0).tolist()
    ci95 = group["ci_95"].fillna(0).tolist()
    risks = group["Risk_Status"].tolist()

    bar_colors = [RISK_COLORS.get(r, DEFAULT_COLOR) for r in risks]
    n = len(group)
    x = np.arange(n)

    fig, ax = plt.subplots(figsize=(5.5, 3.2), facecolor="#0D0D0D")
    ax.set_facecolor("#0D0D0D")

    # Prediction interval band
    for i in range(n):
        if ci95[i] > 0:
            ax.fill_between(
                [i - 0.28, i + 0.28],
                [ci05[i], ci05[i]],
                [ci95[i], ci95[i]],
                color="#AAAAAA",
                alpha=0.25,
                zorder=1,
            )

    # Bars
    ax.bar(x, pct, color=bar_colors, width=0.62, zorder=2, edgecolor="none")

    # Trend line + dots
    if n > 1:
        ax.plot(x, pct, color="white", linewidth=1.1, zorder=3, alpha=0.7)
    ax.scatter(x, pct, color="white", s=22, zorder=4, alpha=0.9)

    # Axes styling
    ax.set_xticks(x)
    ax.set_xticklabels(month_labels, fontsize=7.5, color="#CCCCCC")
    ax.set_ylabel("Streamflow Percentile (%)", fontsize=7.5, color="#BBBBBB")
    valid_ci95 = [v for v in ci95 if v > 0]
    ax.set_ylim(0, max(105, max(valid_ci95) * 1.05) if valid_ci95 else 105)
    ax.tick_params(axis="y", colors="#BBBBBB", labelsize=7)
    ax.tick_params(axis="x", colors="#CCCCCC")
    for spine in ax.spines.values():
        spine.set_edgecolor("#333333")
    ax.yaxis.grid(True, color="#222222", linewidth=0.5, zorder=0)
    ax.set_axisbelow(True)

    # Legend
    seen = {r for r in risks if pd.notna(r)}
    level_order = [
        "Extreme Risk",
        "High Risk",
        "Warning",
        "Watch",
        "Safe",
        "Normal",
        "Wet",
    ]
    patches = [
        mpatches.Patch(facecolor=RISK_COLORS.get(r, DEFAULT_COLOR), label=r)
        for r in level_order
        if r in seen
    ]
    ci_patch = mpatches.Patch(facecolor="#AAAAAA", alpha=0.4, label="5-95% CI")
    leg = ax.legend(
        handles=patches + [ci_patch],
        fontsize=6,
        loc="upper left",
        facecolor="#1A1A1A",
        edgecolor="#444444",
        framealpha=0.85,
        ncol=2,
    )
    for txt in leg.get_texts():
        txt.set_color("#DDDDDD")

    ax.set_title(
        "Monthly Drought Intensity", fontsize=8.5, color="#EEEEEE", pad=6, loc="right"
    )

    plt.tight_layout(pad=0.5)
    buf = io.BytesIO()
    fig.savefig(
        buf, format="png", dpi=130, bbox_inches="tight", facecolor=fig.get_facecolor()
    )
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode("utf-8")


print("Chart builder ready OK")

In [ ]:
# Preview: chart for the first data center
sample_name = monthly["Name"].iloc[0]
sample_grp = monthly[monthly["Name"] == sample_name].reset_index(drop=True)

b64 = make_chart_b64(sample_name, sample_grp)
from IPython.display import HTML

HTML(f'<img src="data:image/png;base64,{b64}" style="background:#0D0D0D;padding:6px">')

In [ ]:
print("Building interactive map (1-3 min) ...")

fmap = folium.Map(
    location=[38.0, -97.0],
    zoom_start=4,
    tiles="CartoDB dark_matter",
    control_scale=True,
)

cluster = MarkerCluster(
    options={"maxClusterRadius": 40, "disableClusteringAtZoom": 9}
).add_to(fmap)

legend_html = """
<div style="
    position: fixed; bottom: 40px; right: 20px; z-index: 9999;
    background: rgba(13,13,13,0.92); border: 1px solid #444;
    border-radius: 8px; padding: 10px 14px; font-family: Arial, sans-serif;
    font-size: 12px; color: #DDD; line-height: 1.7;
">
  <b style='font-size:13px;color:#FFF'>Drought Risk</b><br>
  <span style='color:#8B3A00'>&#9632;</span> Extreme Risk<br>
  <span style='color:#FF6B00'>&#9632;</span> High Risk<br>
  <span style='color:#FFD700'>&#9632;</span> Warning<br>
  <span style='color:#4CAF50'>&#9632;</span> Safe / Normal<br>
  <hr style='border-color:#444;margin:4px 0'>
  <span style='font-size:10px;color:#999'>Click a marker for the monthly chart</span>
</div>
"""
fmap.get_root().html.add_child(folium.Element(legend_html))

dc_meta = df[meta_cols].drop_duplicates("Name").set_index("Name")
grouped = monthly.groupby("Name")
total = len(grouped)

for i, (name, grp) in enumerate(grouped):
    if i % 100 == 0:
        print(f"  {i:>4} / {total}", end="\r")

    meta = dc_meta.loc[name]
    lat = float(meta["Latitude"])
    lon = float(meta["Longitude"])
    risk = worst_risk(grp)
    mcolor = MARKER_COLORS.get(risk, "#888888")

    b64 = make_chart_b64(name, grp.reset_index(drop=True))

    power_str = str(meta.get("Power", "")).strip()
    power_line = (
        f"<b>Capacity:</b> {power_str}<br>" if power_str and power_str != "nan" else ""
    )
    dist_km = f"{float(meta['distance_km']):.1f}"
    station_id = str(meta["nearest_station_id"])
    operator = str(meta.get("Operator", ""))
    city = str(meta.get("City", ""))
    state = str(meta.get("State", ""))

    popup_html = (
        '<div style="font-family:Arial,sans-serif;background:#0D0D0D;color:#DDD;'
        'padding:10px;border-radius:6px;min-width:380px;max-width:560px;">'
        f'<div style="font-size:13px;font-weight:bold;color:#FFF;margin-bottom:4px">{name}</div>'
        f'<div style="font-size:11px;color:#AAA;margin-bottom:6px">{operator} &middot; {city}, {state}</div>'
        f'<div style="font-size:10px;color:#888;margin-bottom:8px">{power_line}'
        f"<b>USGS Gage:</b> {station_id} ({dist_km} km away)<br>"
        f'<b>Overall risk:</b> <span style="color:{mcolor};font-weight:bold">{risk}</span></div>'
        f'<img src="data:image/png;base64,{b64}" style="width:100%;border-radius:4px;display:block">'
        "</div>"
    )

    folium.CircleMarker(
        location=[lat, lon],
        radius=6,
        color=mcolor,
        fill=True,
        fill_color=mcolor,
        fill_opacity=0.80,
        weight=1.2,
        tooltip=f"{name} [{risk}]",
        popup=folium.Popup(popup_html, max_width=580),
    ).add_to(cluster)

print(f"\n{total} markers added.")

In [ ]:
OUT_HTML = "drought_risk_map.html"
fmap.save(OUT_HTML)
print(f"Map saved: {OUT_HTML}")

from IPython.display import IFrame

IFrame(OUT_HTML, width="100%", height="620")